In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
columns = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
           'acceleration', 'model_year', 'origin', 'car_name']

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Auto MPG/auto-mpg.data",
                 sep=r'\s+', header=None, names=columns, na_values='?')
df

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino
...,...,...,...,...,...,...,...,...,...
393,27.0,4,140.0,86.0,2790.0,15.6,82,1,ford mustang gl
394,44.0,4,97.0,52.0,2130.0,24.6,82,2,vw pickup
395,32.0,4,135.0,84.0,2295.0,11.6,82,1,dodge rampage
396,28.0,4,120.0,79.0,2625.0,18.6,82,1,ford ranger


In [4]:
print(df.shape)            # (398, 9)
print(df.isnull().sum())   # horsepower 6개 결측
df.describe()

(398, 9)
mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
car_name        0
dtype: int64


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
count,398.000000,398.000000,398.000000,392.000000,398.000000,398.000000,398.000000,398.000000
mean,23.514573,5.454774,193.425879,104.469388,2970.424623,15.568090,76.010050,1.572864
std,7.815984,1.701004,104.269838,38.491160,846.841774,2.757689,3.697627,0.802055
min,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000,1.000000
25%,17.500000,4.000000,104.250000,75.000000,2223.750000,13.825000,73.000000,1.000000
50%,23.000000,4.000000,148.500000,93.500000,2803.500000,15.500000,76.000000,1.000000
75%,29.000000,8.000000,262.000000,126.000000,3608.000000,17.175000,79.000000,2.000000
max,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,82.000000,3.000000


In [5]:
# 1) 결측치가 있는 6개 행 삭제
df = df.dropna()

# 2) 자동차 이름은 예측에 쓸 수 없는 문자라서 제거
df = df.drop('car_name', axis=1)

# 3) origin(1=미국, 2=유럽, 3=일본)은 숫자지만 크기 의미가 없는 범주라서 원-핫 인코딩
df = pd.get_dummies(df, columns=['origin'], prefix='origin', dtype=int)

print(df.shape)   # (392, 10)
df.head()

(392, 10)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_1,origin_2,origin_3
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,0,0
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,0,0
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,0,0
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,0,0
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,0,0


In [6]:
X = df.drop('mpg', axis=1)
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def evaluate(name, y_test, pred):
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred)
    print(f"[{name}] MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.3f}")

In [7]:
model_dt = DecisionTreeRegressor(random_state=42)
model_dt.fit(X_train, y_train)

pred_dt = model_dt.predict(X_test)
evaluate("Decision Tree", y_test, pred_dt)

[Decision Tree] MAE=2.31  RMSE=3.30  R2=0.786


In [8]:
model_rf = RandomForestRegressor(random_state=42)
model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)
evaluate("Random Forest", y_test, pred_rf)

[Random Forest] MAE=1.72  RMSE=2.40  R2=0.887


In [9]:
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test)
evaluate("Linear Regression", y_test, pred_lr)

[Linear Regression] MAE=2.46  RMSE=3.26  R2=0.792


In [10]:
model_kn = make_pipeline(StandardScaler(), KNeighborsRegressor())
model_kn.fit(X_train, y_train)

pred_kn = model_kn.predict(X_test)
evaluate("KNN", y_test, pred_kn)

[KNN] MAE=2.05  RMSE=2.72  R2=0.855
